In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pymysql
import os
from dotenv import load_dotenv

load_dotenv()

# MySQL Connection
def get_db_connection():
    return pymysql.connect(
        host=os.getenv('MYSQL_HOST'),
        port=int(os.getenv('MYSQL_PORT', 3306)),
        user=os.getenv('MYSQL_USER'),
        password=os.getenv('MYSQL_PASSWORD'),
        database=os.getenv('MYSQL_DATABASE')
    )

In [6]:
conn = get_db_connection()

# Read kuliner data from MySQL
query = '''
SELECT 
    kk.nama_kategori as Kategori,
    k.nama_tempat,
    k.jam_buka,
    k.jam_tutup,
    k.lokasi_geo,
    k.htm_min,
    k.htm_max,
    k.alamat
FROM kuliner k
JOIN kategori_kuliner kk ON k.kategori_kuliner_id = kk.kategori_kuliner_id
'''

df_kuliner = pd.read_sql(query, conn)
conn.close()

# Clean column names
df_kuliner.columns = df_kuliner.columns.str.replace('_', ' ').str.title()

# Rename columns
column_mapping = {
    'Kategori': 'Kategori',
    'Nama Tempat': 'Nama Tempat',
    'Jam Buka': 'Jam Buka',
    'Jam Tutup': 'Jam Tutup',
    'Gps Location': 'GPS Location',
    'Htm Min': 'HTM Min',
    'Htm Max': 'HTM Max',
    'Alamat': 'Alamat'
}

df_kuliner = df_kuliner.rename(columns=column_mapping)

# Add rating column (default 0 for now, will be replaced with actual ratings from DB)
df_kuliner['Rating'] = 0

df_kuliner.head()

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_9688\2073027010.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kuliner = pd.read_sql(query, conn)


,Kategori,Nama Tempat,Jam Buka,Jam Tutup,Lokasi Geo,HTM Min,HTM Max,Alamat,Rating
0,Inovatif,Singosing Cafe & Resto,10:00:00,22:00:00,"-8.3123694, 114.2441926",20000.0,140000.0,"Jl. Aruji Karta Winata, Suruh, juruh, Kec. Sin...",0
1,Inovatif,Warung Seblang,09:00:00,21:00:00,"-8.3007263, 114.2459139",20000.0,40000.0,"Jl. Krajan Timur Desa, Krajan Timur, Padang, ...",0
2,Inovatif,Gen Gentong Resto,10:00:00,22:00:00,"-8.3058414, 114.2169849",25000.0,50000.0,"Tegalmojo, Jl. Aruji Karta Winata, Dusun Kraja...",0
3,Inovatif,Warung Maharrani,10:00:00,22:00:00,"-8.2794869, 114.2485760",10000.0,50000.0,"Jl. Songgon, Cemoro, Singolatren, Kec. Singojuruh",0
4,Inovatif,PanggonMu (Warung PM),00:00:00,22:00:00,"-8.3290854, 114.2528414",20000.0,50000.0,"Padang Bulan, Benelan Kidul, Kec. Singojuruh",0


In [7]:
# Rating sudah dibersihkan di cell sebelumnya, tidak perlu dibersihkan lagi
df_kuliner.head()

,Kategori,Nama Tempat,Jam Buka,Jam Tutup,Lokasi Geo,HTM Min,HTM Max,Alamat,Rating
0,Inovatif,Singosing Cafe & Resto,10:00:00,22:00:00,"-8.3123694, 114.2441926",20000.0,140000.0,"Jl. Aruji Karta Winata, Suruh, juruh, Kec. Sin...",0
1,Inovatif,Warung Seblang,09:00:00,21:00:00,"-8.3007263, 114.2459139",20000.0,40000.0,"Jl. Krajan Timur Desa, Krajan Timur, Padang, ...",0
2,Inovatif,Gen Gentong Resto,10:00:00,22:00:00,"-8.3058414, 114.2169849",25000.0,50000.0,"Tegalmojo, Jl. Aruji Karta Winata, Dusun Kraja...",0
3,Inovatif,Warung Maharrani,10:00:00,22:00:00,"-8.2794869, 114.2485760",10000.0,50000.0,"Jl. Songgon, Cemoro, Singolatren, Kec. Singojuruh",0
4,Inovatif,PanggonMu (Warung PM),00:00:00,22:00:00,"-8.3290854, 114.2528414",20000.0,50000.0,"Padang Bulan, Benelan Kidul, Kec. Singojuruh",0


In [8]:
# Read ratings from MySQL
conn = get_db_connection()

query = '''
SELECT 
    user_id,
    k.nama_tempat as kuliner,
    rk.nilai_rating as rating
FROM rating_kuliner rk
JOIN kuliner k ON rk.kuliner_id = k.kuliner_id
ORDER BY user_id
'''

ratings_kuliner = pd.read_sql(query, conn)

# Fallback: generate mock data if no ratings in DB
if len(ratings_kuliner) == 0:
    ratings_kuliner = pd.DataFrame({
        'user_id': np.random.randint(1, 15, 80),
        'kuliner': np.random.choice(df_kuliner['Nama Tempat'], 80),
        'rating': np.random.randint(3, 6, 80)
    })

conn.close()
ratings_kuliner

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_9688\1702842330.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ratings_kuliner = pd.read_sql(query, conn)


,user_id,kuliner,rating
0,8,Camping resto,3
1,1,Kampung Lobster Banyuwangi,3
2,9,Balesaji,4
3,12,D'calma Cafe Resto,3
4,14,TherMose,5
...,...,...,...
75,2,Senja. co cafe n resto,5
76,10,Dapoer thata kemiren,5
77,3,Nona Restoran,4
78,10,Warung suku osing,3


In [9]:
# Read favorit from MySQL
conn = get_db_connection()

query = '''
SELECT 
    user_id,
    k.nama_tempat as kuliner,
    1 as saved
FROM favorit_kuliner fk
JOIN kuliner k ON fk.kuliner_id = k.kuliner_id
ORDER BY user_id
'''

saved_kuliner = pd.read_sql(query, conn)

# Fallback: generate mock data if no favorit in DB
if len(saved_kuliner) == 0:
    saved_kuliner = pd.DataFrame({
        'user_id': np.random.randint(1, 15, 40),
        'kuliner': np.random.choice(df_kuliner['Nama Tempat'], 40),
        'saved': 1
    })

conn.close()
saved_kuliner

C:\Users\mallexibra\AppData\Local\Temp\ipykernel_9688\970136448.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  saved_kuliner = pd.read_sql(query, conn)


,user_id,kuliner,saved
0,4,Ujung Jawa cafe and eatery,1
1,12,Rodjo Nogo,1
2,8,Maskot Resto,1
3,9,Hedon Estate Banyuwangi,1
4,4,Betutu Tepi Sawah,1
5,1,Kichi Japanese Food,1
6,10,Kembang langit Traditional Restoran & Cafe,1
7,3,PanggonMu (Warung PM),1
8,9,Pawon Mbok Pesek,1
9,3,Dapur Ijen,1


In [10]:
interaction_kuliner = pd.merge(
    ratings_kuliner,
    saved_kuliner,
    on=['user_id','kuliner'],
    how='outer'
).fillna(0)

interaction_kuliner['score'] = (
    interaction_kuliner['rating'] +
    interaction_kuliner['saved']
)

In [11]:
user_item_kuliner = interaction_kuliner.pivot_table(
    index='user_id',
    columns='kuliner',
    values='rating'
).fillna(0)

user_item_kuliner.head()

kuliner,Alas Restaurant,Baba Cafe & Resto,Balesaji,Bebek Songkem Taretan,Berco,Betutu Tepi Sawah,Bibiba Cafe,Cafe Resto & Glamping Pinggir Lepen,Camp Resto and Caffe,Camping resto,...,Warung 99 Rogojampi Cabang 2 Kedaleman,Warung Dalam Negeri,Warung Maharrani,Warung Pakdhe Panjang Jiwo BWI,Warung Pondok Riko,Warung Seblang,Warung kali sodong,Warung suku osing,Wedangan Gebyok,pelabuhan rakyat pantai boom
user_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# Optimasi: gunakan sparse matrix untuk similarity
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# Convert ke sparse matrix untuk efisiensi memori dan komputasi
sparse_matrix = csr_matrix(user_item_kuliner.values)

similarity_kuliner = cosine_similarity(sparse_matrix)

similarity_kuliner_df = pd.DataFrame(
    similarity_kuliner,
    index=user_item_kuliner.index,
    columns=user_item_kuliner.index
)

In [13]:
def get_cf_kuliner(user_id):

    similar_users = similarity_kuliner_df[user_id]\
        .sort_values(ascending=False)[1:6]

    similar_users_index = similar_users.index

    rekomendasi = interaction_kuliner[
        interaction_kuliner['user_id'].isin(similar_users_index)
    ]

    rekomendasi = rekomendasi.sort_values(
        by='rating',
        ascending=False
    )

    return rekomendasi['kuliner'].unique()

In [14]:
def content_based_kuliner(
    kuliner_list,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    filtered = df_kuliner[
        (df_kuliner['Nama Tempat'].isin(kuliner_list)) &
        (df_kuliner['Kategori'].isin(kategori)) &
        (df_kuliner['HTM Min'] >= budget_min) &
        (df_kuliner['HTM Max'] <= budget_max) &
        (df_kuliner['Rating'] >= rating_min)
    ]

    return filtered

In [15]:
def hybrid_kuliner(
    user_id,
    kategori,
    budget_min,
    budget_max,
    rating_min
):

    cf_result = get_cf_kuliner(user_id)

    # fallback jika CF sedikit
    if len(cf_result) < 5:
        cf_result = df_kuliner['Nama Tempat']

    final = content_based_kuliner(
        cf_result,
        kategori,
        budget_min,
        budget_max,
        rating_min
    )

    final = final.sort_values(
        by='Rating',
        ascending=False
    )

    return final

In [16]:
hasil_kuliner = hybrid_kuliner(
    user_id=8,
    kategori=['Inovatif', 'Tradisional'],
    budget_min=0,
    budget_max=100000,
    rating_min=4.5
)

print(hasil_kuliner[['Nama Tempat','Kategori', 'HTM Min', 'HTM Max', 'Rating']])

Empty DataFrame
Columns: [Nama Tempat, Kategori, HTM Min, HTM Max, Rating]
Index: []


In [17]:
cf_result = get_cf_kuliner(2)
cf_result

<StringArray>
[                                 'Mie Ayam Purwo',
                            'Rumah Makan Pak Poer',
      'Kembang langit Traditional Restoran & Cafe',
                         'Trilogi Cafe  and Resto',
                                'Waroeng Kemarang',
                             'Warung Dalam Negeri',
                       'Rujak kelang (Neng Licha)',
                                'Resto Legian BWI',
                                      'Rodjo Nogo',
                      'Kampung Lobster Banyuwangi',
                    'Ikan Bakar Pesona Banyuwangi',
                     'RM. Sate Kambing Cak Bagong',
                                   'Nona Restoran',
                      'Casabanyu Restaurant & Bar',
                     'Kopi Bukit Osing Wonderland',
                       'Rumah Makan Duta Kalibaru',
                            'Java River Side Cafe',
                            'Urup-urup Cafe Resto',
                             'Resto Van De Oasing'